In [ ]:
# ==========================================================
# Maestría en Ciencia y Análisis de Datos
# Universidad Mayor de San Andrés
# ----------------------------------------------------------
#   Modelos lineales y modelos lineales generalizados
# ----------------------------------------------------------
#        Rolando Gonzales Martinez, Julio 2026
# ==========================================================
#       Modelos lineales multiecuacionales: VAR(p)
# ==========================================================

import matplotlib.pyplot as plt
import pandas as pd

# Importar el conjunto de datos desde GitHub
url = 'https://raw.githubusercontent.com/rogon666/UMSA/refs/heads/main/2026/MLMLG_2026/datos/oil_prices.csv'

# Cargar los datos en un DataFrame
data = pd.read_csv(url)

# Convertir la fecha
data["date"] = pd.to_datetime(data["date"], format="%b-%y")

# Ordenar cronológicamente
data = data.sort_values("date").reset_index(drop=True)

# Establecer la fecha como índice
data = data.set_index("date")

print(data.head())


In [ ]:
# ==========================================================
# Graficar precios y variaciones porcentuales
# ==========================================================

data[["Brent", "WTI"]].plot(figsize=(11, 5))

plt.title("Precios mensuales del petróleo")
plt.xlabel("Fecha")
plt.ylabel("Dólares por barril")
plt.grid(True)
plt.tight_layout()
plt.show()
variaciones = data[["Brent", "WTI"]].pct_change() * 100

# Eliminar la primera observación, que contiene valores NaN
variaciones = variaciones.dropna()

variaciones.columns = ["dBrent", "dWTI"]

print(variaciones.head())
variaciones.plot(figsize=(11, 5))

plt.axhline(y=0, color="black", linestyle="--")

plt.title("Variaciones porcentuales mensuales")
plt.xlabel("Fecha")
plt.ylabel("Variación porcentual (%)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------
# Seleccion de rezagos p en el modelo VAR(p)
# ------------------------------------------
from statsmodels.tsa.api import VAR

modelo = VAR(variaciones)

seleccion_rezagos = modelo.select_order(maxlags=12)

print(seleccion_rezagos.summary())

In [ ]:
# ---------------------------------------
# Estimacion del modelo VAR(p)
# ---------------------------------------
orden_p_VAR = 2
resultado_var = modelo.fit(orden_p_VAR)

print(resultado_var.summary())

In [ ]:
# --------------------------------------------------
# Causalidad de Granger
# --------------------------------------------------
causalidad_brent_wti = resultado_var.test_causality(
    caused="dWTI",
    causing=["dBrent"],
    kind="f"
)

print(causalidad_brent_wti.summary())


causalidad_wti_brent = resultado_var.test_causality(
    caused="dBrent",
    causing=["dWTI"],
    kind="f"
)
print(causalidad_wti_brent.summary())

In [ ]:
# Pronosticos

horizonte = 12

# Últimas p observaciones requeridas por el VAR
datos_iniciales = variaciones.values[-resultado_var.k_ar:]

pronostico = resultado_var.forecast(
    y=datos_iniciales,
    steps=horizonte
)

# Fechas futuras mensuales
fechas_futuras = pd.date_range(
    start=variaciones.index[-1] + pd.offsets.MonthBegin(1),
    periods=horizonte,
    freq="MS"
)

pronostico_df = pd.DataFrame(
    pronostico,
    index=fechas_futuras,
    columns=["Pronostico_dBrent", "Pronostico_dWTI"]
)

print(pronostico_df)

# Graficos

plt.figure(figsize=(11, 5))

plt.plot(
    variaciones.index,
    variaciones["dBrent"],
    label="Brent observado"
)

plt.plot(
    pronostico_df.index,
    pronostico_df["Pronostico_dBrent"],
    linestyle="--",
    label="Brent pronosticado"
)

plt.axhline(y=0, color="black", linestyle=":")
plt.title("Pronóstico VAR de la variación porcentual del Brent")
plt.xlabel("Fecha")
plt.ylabel("Variación porcentual (%)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))

plt.plot(
    variaciones.index,
    variaciones["dWTI"],
    label="WTI observado"
)

plt.plot(
    pronostico_df.index,
    pronostico_df["Pronostico_dWTI"],
    linestyle="--",
    label="WTI pronosticado"
)

plt.axhline(y=0, color="black", linestyle=":")
plt.title("Pronóstico VAR de la variación porcentual del WTI")
plt.xlabel("Fecha")
plt.ylabel("Variación porcentual (%)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------------------------
# Convertir los pronósticos porcentuales a niveles
# ----------------------------------------------------------

# Últimos precios observados
ultimo_brent = data["Brent"].iloc[-1]
ultimo_wti = data["WTI"].iloc[-1]

# Factores de crecimiento pronosticados
factor_brent = 1 + pronostico_df["Pronostico_dBrent"] / 100
factor_wti = 1 + pronostico_df["Pronostico_dWTI"] / 100

# Reconstruir los niveles mediante acumulación compuesta
pronostico_df["Pronostico_Brent"] = ultimo_brent * factor_brent.cumprod()
pronostico_df["Pronostico_WTI"] = ultimo_wti * factor_wti.cumprod()

print(pronostico_df)

# ----------------------------------------------------------
# Pronóstico de Brent en niveles
# ----------------------------------------------------------

plt.figure(figsize=(11, 5))

plt.plot(
    data.index,
    data["Brent"],
    label="Brent observado"
)

plt.plot(
    pronostico_df.index,
    pronostico_df["Pronostico_Brent"],
    linestyle="--",
    label="Brent pronosticado"
)

plt.title("Pronóstico VAR del precio del Brent")
plt.xlabel("Fecha")
plt.ylabel("Dólares por barril")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------
# Pronóstico de WTI en niveles
# ----------------------------------------------------------

plt.figure(figsize=(11, 5))

plt.plot(
    data.index,
    data["WTI"],
    label="WTI observado"
)

plt.plot(
    pronostico_df.index,
    pronostico_df["Pronostico_WTI"],
    linestyle="--",
    label="WTI pronosticado"
)

plt.title("Pronóstico VAR del precio del WTI")
plt.xlabel("Fecha")
plt.ylabel("Dólares por barril")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))

plt.plot(
    data.index,
    data["Brent"],
    label="Brent observado"
)

plt.plot(
    data.index,
    data["WTI"],
    label="WTI observado"
)

plt.plot(
    pronostico_df.index,
    pronostico_df["Pronostico_Brent"],
    linestyle="--",
    label="Brent pronosticado"
)

plt.plot(
    pronostico_df.index,
    pronostico_df["Pronostico_WTI"],
    linestyle="--",
    label="WTI pronosticado"
)

plt.title("Pronósticos VAR de Brent y WTI en niveles")
plt.xlabel("Fecha")
plt.ylabel("Dólares por barril")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()